# Question 2 — Potential Outcomes, Causal Graphs, and SEM (LaLonde)

We use the LaLonde job training dataset to study whether participation in a job training program affected workers' real earnings in 1978.

Only `numpy` and `pandas` are used for the code.

In [35]:
import numpy as np
import pandas as pd

df = pd.read_csv('lalonde_data.csv')
df.head()

,ID,treat,age,educ,black,hispan,married,nodegree,re74,re75,re78
0,NSW1,1,37,11,1,0,1,1,0.0,0.0,9930.0460
1,NSW2,1,22,9,0,1,0,1,0.0,0.0,3595.8940
2,NSW3,1,30,12,1,0,0,0,0.0,0.0,24909.4500
3,NSW4,1,27,11,1,0,0,1,0.0,0.0,7506.1460
4,NSW5,1,33,8,1,0,0,1,0.0,0.0,289.7899


In [36]:
print('Shape:', df.shape)
print(df['treat'].value_counts())

Shape: (614, 11)
treat
0    429
1    185
Name: count, dtype: int64


## Q2.1 — Treatment and outcome variables

- **Treatment**: `treat`. Equals 1 if the worker participated in the job training program, 0 otherwise.
- **Outcome**: `re78`. Real earnings of the worker in 1978 (in dollars).

We want to know whether receiving the training (`treat = 1`) caused a change in 1978 earnings (`re78`).

## Q2.2 — Potential outcomes

For each worker $i$:

- $Y_i(1)$ = the 1978 earnings worker $i$ would have **if** they received the training.
- $Y_i(0)$ = the 1978 earnings worker $i$ would have **if** they did not receive the training.

The **individual treatment effect** is:
$$
\tau_i = Y_i(1) - Y_i(0).
$$

## Q2.3 — Why we cannot observe both $Y_i(1)$ and $Y_i(0)$

Each worker is either trained or not trained. We observe only one of the two potential outcomes for each person. The other one is counterfactual , what would have happened in a world that did not occur. This is the **fundamental problem of causal inference**.

## Q2.4 — Average Treatment Effect (ATE)

$$
\text{ATE} = \mathbb{E}[Y_i(1) - Y_i(0)] = \mathbb{E}[Y_i(1)] - \mathbb{E}[Y_i(0)].
$$

It is the average causal effect of the training over the population.

## Q2.5 — Naive difference in means

$$
\widehat{\text{ATE}}_{\text{naive}} = \bar{Y}_{\text{treated}} - \bar{Y}_{\text{control}}.
$$

In [37]:
y_treated = df.loc[df['treat'] == 1, 're78'].mean()
y_control = df.loc[df['treat'] == 0, 're78'].mean()

ate_naive = y_treated - y_control

print(f'Mean re78 treated : {y_treated:.2f}')
print(f'Mean re78 control : {y_control:.2f}')
print(f'Naive ATE         : {ate_naive:.2f}')

Mean re78 treated : 6349.14
Mean re78 control : 6984.17
Naive ATE         : -635.03


## Q2.6 — Interpretation of the naive estimate

The naive estimate is **negative**, about $635$ dollars. Taken at face value, it would suggest that workers who received the training earned **less** in 1978 than workers who did not. This is counter-intuitive: a useful training program is not expected to lower earnings.

## Q2.7 — Why the naive difference can fail

The naive estimate compares treated and untreated workers as if treatment was randomly assigned. But it was not. Workers in the program were selected (or self-selected) into it, and they look very different from the controls on pre-treatment variables.

Example using this dataset: among the treated workers, a much higher share are Black, single, and have no high school degree, and their 1974 and 1975 earnings (`re74`, `re75`) are much lower than those of controls. So the comparison mixes two effects:

1. The actual effect of training.
2. The fact that treated workers were already more disadvantaged before training.

We check this below.

In [38]:
cols = ['age','educ','black','hispan','married','nodegree','re74','re75']
df.groupby('treat')[cols].mean().T

treat,0,1
age,28.030303,25.816216
educ,10.235431,10.345946
black,0.202797,0.843243
hispan,0.142191,0.059459
married,0.512821,0.189189
nodegree,0.596737,0.708108
re74,5619.236506,2095.573689
re75,2466.484443,1532.055314


## Q2.8 — Guessed causal graph

We treat `age`, `educ`, `black`, `hispan` as background (exogenous) characteristics of the worker. They influence `married`, `nodegree`, past earnings `re74`, `re75`, the chance of entering the training program `treat`, and finally `re78`.

The DAG is given below as an explicit edge list (`A -> B` means "A causes B").

```
Background covariates (exogenous):
  age, educ, black, hispan

Directed edges:
  age, educ, black, hispan        ->  married
  age, educ, black, hispan        ->  nodegree
  age, educ, black, hispan        ->  re74
  age, educ, black, hispan        ->  re75

  age, educ, black, hispan,
  married, nodegree, re74, re75   ->  treat

  treat                           ->  re78
  age, educ, black, hispan,
  married, nodegree, re74, re75   ->  re78
```

**Reasoning.** Demographics (age, education, race) are determined first. They influence family status (`married`), education completion (`nodegree`), and past earnings (`re74`, `re75`). All of these — demographics and pre-treatment outcomes — then influence who selects into the program (`treat`). Both the treatment and the pre-treatment variables affect 1978 earnings (`re78`); the direct arrow `treat -> re78` is the causal effect we want to estimate.

## Q2.9 — Structural Equation Model (SEM)

Let $T = \text{treat}$ and $Y = \text{re78}$. The pre-treatment covariates are $X = (\text{age}, \text{educ}, \text{black}, \text{hispan}, \text{married}, \text{nodegree}, \text{re74}, \text{re75})$.

**Treatment equation** (linear probability model for participation):
$$
T_i = \alpha_0 + \alpha_1\,\text{age}_i + \alpha_2\,\text{educ}_i + \alpha_3\,\text{black}_i + \alpha_4\,\text{hispan}_i
     + \alpha_5\,\text{married}_i + \alpha_6\,\text{nodegree}_i + \alpha_7\,\text{re74}_i + \alpha_8\,\text{re75}_i + u_i.
$$

**Outcome equation**:
$$
Y_i = \beta_0 + \beta_1\,T_i + \beta_2\,\text{age}_i + \beta_3\,\text{educ}_i + \beta_4\,\text{black}_i + \beta_5\,\text{hispan}_i
     + \beta_6\,\text{married}_i + \beta_7\,\text{nodegree}_i + \beta_8\,\text{re74}_i + \beta_9\,\text{re75}_i + \varepsilon_i.
$$

The coefficient of interest is $\beta_1$, the effect of training on 1978 earnings, holding the pre-treatment variables fixed.

## Q2.10 — OLS estimation of the outcome equation

We use the closed-form OLS estimator:
$$
\hat{\beta} = (X^\top X)^{-1} X^\top y.
$$

In [39]:
y = df['re78'].values
regressors = ['treat','age','educ','black','hispan','married','nodegree','re74','re75']
X = df[regressors].values
X = np.column_stack([np.ones(len(y)), X])

beta_hat = np.linalg.inv(X.T @ X) @ X.T @ y

names = ['intercept'] + regressors
coefs = pd.Series(beta_hat, index=names)
coefs.round(3)

intercept      66.515
treat        1548.244
age            12.978
educ          403.941
black       -1240.644
hispan        498.897
married       406.621
nodegree      259.817
re74            0.296
re75            0.232
dtype: float64

In [40]:
print(f'Estimated effect of training (beta1): {coefs["treat"]:.2f}')

Estimated effect of training (beta1): 1548.24


**Meaning of $\beta_1$**: the average change in 1978 earnings associated with receiving the training, **holding the pre-treatment variables fixed**. If the linear model and the chosen controls are correct, $\beta_1$ is the regression-adjusted estimate of the ATE.

## Q2.11 — Naive vs OLS-adjusted estimate

In [41]:
print(f'Naive ATE    : {ate_naive:.2f}')
print(f'OLS-adjusted : {coefs["treat"]:.2f}')

Naive ATE    : -635.03
OLS-adjusted : 1548.24


The two estimates differ a lot, and they even have different signs.

The naive estimate is negative because treated workers were already more disadvantaged before the program (lower past earnings, less education, more often without a high school degree). The naive comparison confuses the training effect with these pre-existing differences.

The OLS estimate controls for these pre-treatment differences. Once we condition on `age`, `educ`, `black`, `hispan`, `married`, `nodegree`, `re74`, `re75`, the estimated effect of training on 1978 earnings becomes positive (about +1548 dollars).

## Q2.12 — Does a positive coefficient prove causation?

No. A positive $\hat{\beta}_1$ is only a causal estimate if several assumptions hold:

1. **Unconfoundedness**: every variable that affects both `treat` and `re78` is included as a control. If important variables are missing (motivation, ability, local labor market conditions), the estimate is biased.
2. **Correct functional form**: the relationship between covariates and `re78` is linear and additive as written. If it is not, $\hat{\beta}_1$ absorbs misspecification.
3. **Overlap**: treated and untreated workers must look similar enough on the covariates that the comparison is meaningful.

So OLS gives a better estimate than the naive difference, but it is not a proof of causation. The only fully reliable way to identify the ATE would be a randomized assignment of training.